# Lab 2 — Imbalance Analysis

**Day 06 · Anomaly Detection · Cisco AI/ML Training**

---

## Learning objectives

1. Quantify **class imbalance** (990 legit vs 10 fraud).
2. Train a **majority-class baseline** (`DummyClassifier`).
3. Show **99% accuracy** can still mean **0** fraud detections.
4. Introduce **precision, recall, F1** for rare-event evaluation.

> **Checkpoints:** imbalance **99:1** · baseline accuracy **0.99** · baseline F1 (fraud) = **0.00**



## Why this matters

<!-- cisco-enrich-2026-06 -->

Fraud, churn, and failure prediction are **rare-event** problems. Product owners watch accuracy; risk teams need **recall** on the minority class. This lab builds the vocabulary you will use through Day 6 capstone.


## The accuracy trap

| Metric | Majority baseline | Problem |
|--------|-------------------|--------|
| **Accuracy** | **0.99** | Looks great — always predicts legit |
| **F1 (fraud)** | **0.00** | Catches zero fraud cases |

With **1%** fraud rate, predicting "not fraud" every time is 99% accurate but useless for detection.

---

## 1. Load features and target

In [ ]:
%matplotlib inline

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

GH_ROOT = Path.cwd().resolve()
if GH_ROOT.name == "notebooks":
    GH_ROOT = GH_ROOT.parents[2]
elif GH_ROOT.name == "day-06":
    GH_ROOT = GH_ROOT.parents[1]
else:
    for parent in [GH_ROOT, *GH_ROOT.parents]:
        if (parent / "data" / "credit-card" / "credit_card_transactions.csv").is_file():
            GH_ROOT = parent
            break

NUMERIC_FEATURES = ["amount", "distance_from_home"]
CATEGORICAL_FEATURES = ["merchant_category"]

df = pd.read_csv(GH_ROOT / "data" / "credit-card" / "credit_card_transactions.csv")
X = df[NUMERIC_FEATURES + CATEGORICAL_FEATURES]
y = df["is_fraud"]

fraud_count = int(y.sum())
legit_count = len(y) - fraud_count
imbalance_ratio = legit_count / max(fraud_count, 1)

print("Lab 2 — Imbalance analysis")
print(f"total rows: {len(y)}")
print(f"fraud: {fraud_count}, legit: {legit_count}")
print(f"imbalance ratio (legit:fraud): {imbalance_ratio:.1f}:1")
print(f"fraud rate: {y.mean():.4f}")

---

## 2. Class distribution

In [ ]:
fig, ax = plt.subplots(figsize=(5, 4))
counts = y.value_counts().sort_index()
sns.barplot(x=counts.index.map({0: "legit", 1: "fraud"}), y=counts.values, ax=ax, palette="Set2")
ax.set_ylabel("count")
ax.set_title("Class imbalance (full dataset)")
plt.tight_layout()
plt.show()

---

## 3. Stratified train/test split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

split_summary = pd.DataFrame({
    "split": ["train", "test"],
    "total": [len(y_train), len(y_test)],
    "fraud": [int(y_train.sum()), int(y_test.sum())],
    "legit": [int((y_train == 0).sum()), int((y_test == 0).sum())],
})
display(split_summary)

Test set has only **2** fraud cases — metrics will be noisy; say this when interpreting Labs 3–6.

---

## 4. Majority-class baseline

In [ ]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), NUMERIC_FEATURES),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ]
)

baseline = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("clf", DummyClassifier(strategy="most_frequent")),
    ]
)
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)

baseline_acc = (y_pred == y_test).mean()
f1 = f1_score(y_test, y_pred, zero_division=0)

print(f"baseline accuracy (majority class): {baseline_acc:.4f}")
print(f"baseline F1 (fraud): {f1:.4f}")
print(f"unique predictions: {sorted(set(y_pred.tolist()))}")

---

## 5. Classification report

In [ ]:
print(classification_report(y_test, y_pred, zero_division=0))

Focus on the **fraud (1)** row — precision, recall, and F1 are all **0.00**.

---

## 6. Metrics cheat sheet

In [ ]:
metrics_help = pd.DataFrame({
    "metric": ["accuracy", "precision (fraud)", "recall (fraud)", "F1 (fraud)"],
    "baseline": [f"{baseline_acc:.4f}", "0.00", "0.00", f"{f1:.4f}"],
    "higher_better": ["yes", "yes", "yes", "yes"],
    "use_when": [
        "balanced classes only",
        "false alarms costly",
        "missing fraud costly",
        "balance precision & recall",
    ],
})
display(metrics_help)

---

## 7. Checkpoint summary

In [ ]:
assert fraud_count == 10
assert abs(imbalance_ratio - 99.0) < 0.1
assert abs(baseline_acc - 0.99) < 0.01
assert f1 == 0.0
assert int(y_test.sum()) == 2
print("✓ All checkpoint assertions passed")

## Extension — cost of errors (worksheet)

In [ ]:
# hypothetical business costs (USD) — discuss with your table
cost_fn = 5000   # missed fraud
cost_fp = 50     # false alarm review
n_fraud_test = int(y_test.sum())
n_legit_test = int((y_test == 0).sum())
# baseline catches 0 fraud
baseline_cost = n_fraud_test * cost_fn
print(f"baseline missed-fraud cost (upper bound): ${baseline_cost:,}")
print("If a model catches 1 of 2 frauds, how does that change the story?")


---

## Reflection questions

1. Why is accuracy misleading when fraud is 1% of rows?
2. For a bank, is false alarm or missed fraud usually worse?
3. What will resampling (Lab 3) try to fix?

**Previous:** [Lab 1 — Outlier exploration](lab01_outlier_exploration.ipynb)  
**Next:** [Lab 3 — Resampling](lab03_resampling_lab.ipynb)